In [ ]:
!pwd

/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16


In [ ]:
import os
os.chdir('/content/drive/MyDrive/LUNA16/Lung-nodule-detection-LUNA-16')

In [ ]:
!pip install segmentation-models-pytorch
!pip install SimpleITK

In [ ]:
import numpy as np
import pandas as pd
from glob import glob
import os
import torch
import SimpleITK as sitk
# from SUMNet_bn import SUMNet
from torchvision import transforms
import torch.nn.functional as F
import cv2
from tqdm import tqdm_notebook as tq
import segmentation_models_pytorch as smp


In [ ]:
def load_itk_image(filename):
    itkimage = sitk.ReadImage(filename)
    numpyImage = sitk.GetArrayFromImage(itkimage)

    numpyOrigin = np.array(list(reversed(itkimage.GetOrigin())))
    numpySpacing = np.array(list(reversed(itkimage.GetSpacing())))
    return numpyImage, numpyOrigin, numpySpacing

In [ ]:
seg_model_loadPath = './Results/Unet/Adam_1e-4_ep100_CE+Lov/'
netS = smp.Unet(
    encoder_name="resnet34",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=2                       # model output channels (number of classes in your dataset)
)
netS.load_state_dict(torch.load(seg_model_loadPath+'sumnet_best.pt', map_location=torch.device('cpu')))
netS = netS.cuda() # This line is removed to run on CPU, or uncomment and ensure GPU runtime is enabled.
apply_norm = transforms.Normalize([-464.62011798179316],[444.2257740178454])

In [ ]:
cand_path = "./dataset/candidates.csv"
b_sz = 8
df_node = pd.read_csv(cand_path)
subset = ['3']#,'5']
running_correct = 0
count = 0

orig_list = []
pred_list = []
for s in subset:
    print('Subset:',s)
    luna_subset_path = './dataset/subset'+str(s)+'/'
    all_files = os.listdir(luna_subset_path)
    mhd_files = []
    for f in all_files:
        if '.mhd' in f:
            mhd_files.append(f)
    count = 0
    for m in tq(mhd_files):
        mini_df = df_node[df_node["seriesuid"]==m[:-4]]
        itk_img = sitk.ReadImage(luna_subset_path+m)
        img_array = sitk.GetArrayFromImage(itk_img)
        origin = np.array(itk_img.GetOrigin())      # x,y,z  Origin in world coordinates (mm)
        spacing = np.array(itk_img.GetSpacing())
        slice_list = []
        if len(mini_df)>0:
            for i in range(len(mini_df)):
                fName = mini_df['seriesuid'].values[i]
                z_coord = mini_df['coordZ'].values[i]
                orig_class = mini_df['class'].values[i]
                pred = 0
                v_center =np.rint((z_coord-origin[2])/spacing[2])
                img_slice = img_array[int(v_center)]
                mid_mean = img_slice[100:400,100:400].mean()
                img_slice[img_slice==img_slice.min()] = mid_mean
                img_slice[img_slice==img_slice.max()] = mid_mean
                img_slice_tensor = torch.from_numpy(img_slice).unsqueeze(0).float()
                img_slice_norm = apply_norm(img_slice_tensor).unsqueeze(0)

                out = F.softmax(netS(img_slice_norm.cuda()),dim=1)
                out_np = np.asarray(out[0,1].squeeze(0).detach().cpu().numpy()*255,dtype=np.uint8)

                ret, thresh = cv2.threshold(out_np,0,1,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
                connectivity = 4
                output = cv2.connectedComponentsWithStats(thresh, connectivity, cv2.CV_32S)
                stats = output[2]
                temp = stats[1:, cv2.CC_STAT_AREA]
                if len(temp)>0:
                    largest_label = 1 + np.argmax(temp)
                    areas = stats[1:, cv2.CC_STAT_AREA]
                    max_area = np.max(areas)
                    if max_area>150:
                        pred = 1
                if pred == orig_class:
                    running_correct += 1
                pred_list.append(pred)
                orig_list.append(orig_class)
                count += 1

Subset: 3


/tmp/ipython-input-2085256343.py:19: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for m in tq(mhd_files):


  0%|          | 0/89 [00:00<?, ?it/s]

In [ ]:
cand_path = "./dataset/candidates.csv"
b_sz = 8
df_node = pd.read_csv(cand_path)
subset = ['5']#,'5']
running_correct = 0
count = 0

orig_list = []
pred_list = []
for s in subset:
    print('Subset:',s)
    luna_subset_path = './dataset/subset'+str(s)+'/'
    all_files = os.listdir(luna_subset_path)
    mhd_files = []
    for f in all_files:
        if '.mhd' in f:
            mhd_files.append(f)
    count = 0
    for m in tq(mhd_files):
        mini_df = df_node[df_node["seriesuid"]==m[:-4]]
        itk_img = sitk.ReadImage(luna_subset_path+m)
        img_array = sitk.GetArrayFromImage(itk_img)
        origin = np.array(itk_img.GetOrigin())      # x,y,z  Origin in world coordinates (mm)
        spacing = np.array(itk_img.GetSpacing())
        slice_list = []
        if len(mini_df)>0:
            for i in range(len(mini_df)):
                fName = mini_df['seriesuid'].values[i]
                z_coord = mini_df['coordZ'].values[i]
                orig_class = mini_df['class'].values[i]
                pred = 0
                v_center =np.rint((z_coord-origin[2])/spacing[2])
                img_slice = img_array[int(v_center)]
                mid_mean = img_slice[100:400,100:400].mean()
                img_slice[img_slice==img_slice.min()] = mid_mean
                img_slice[img_slice==img_slice.max()] = mid_mean
                img_slice_tensor = torch.from_numpy(img_slice).unsqueeze(0).float()
                img_slice_norm = apply_norm(img_slice_tensor).unsqueeze(0)

                out = F.softmax(netS(img_slice_norm.cuda()),dim=1)
                out_np = np.asarray(out[0,1].squeeze(0).detach().cpu().numpy()*255,dtype=np.uint8)

                ret, thresh = cv2.threshold(out_np,0,1,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
                connectivity = 4
                output = cv2.connectedComponentsWithStats(thresh, connectivity, cv2.CV_32S)
                stats = output[2]
                temp = stats[1:, cv2.CC_STAT_AREA]
                if len(temp)>0:
                    largest_label = 1 + np.argmax(temp)
                    areas = stats[1:, cv2.CC_STAT_AREA]
                    max_area = np.max(areas)
                    if max_area>150:
                        pred = 1
                if pred == orig_class:
                    running_correct += 1
                pred_list.append(pred)
                orig_list.append(orig_class)
                count += 1

Subset: 5


/tmp/ipython-input-1086607081.py:19: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for m in tq(mhd_files):


  0%|          | 0/89 [00:00<?, ?it/s]

In [ ]:
print('Accuarcy:',(running_correct/count)*100)


Accuarcy: 22.260075511458425


In [ ]:
from sklearn.metrics import confusion_matrix
cf = confusion_matrix(orig_list, pred_list)
tn, fp, fn, tp = cf.ravel()

In [ ]:
sensitivity = tp/(tp+fn)
print('Sensitivity:',sensitivity)

Sensitivity: 0.9464285714285714


In [ ]:
specificity = tn/(tn+fp)
print('Specificity:',specificity)

Specificity: 0.22117431773793395


In [ ]:
precision = tp/(tp+fp)
print('Precision:',precision)

Precision: 0.002389055421578129
